In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("NER_master_landslide_dataset_v2.csv")

In [3]:
df.head()

,source_name,source_link,event_id,event_date,event_time,event_title,event_description,location_description,location_accuracy,landslide_category,...,last_edited_date,longitude,latitude,rainfall_1d,rainfall_3d,rainfall_7d,rainfall_14d,max_rainfall_7d,elevation_m,slope_deg
0,E-Pao,http://www.e-pao.net/GP.asp?src=28..210810.aug10,2164,2010-07-29 23:00:00,NaN,"Chingai-Kuingai(Quingai) route (16 kms), Ching...",Since the Chingai-Quingai road has been sunk d...,"Chingai-Kuingai(Quingai) route (16 kms), Ching...",25km,landslide,...,02/15/2018 03:51:00 PM,94.482147,25.309863,17.06,32.41,58.35,101.04,17.06,1552.0,29.263044
1,The Hindu,http://www.thehindu.com/news/national/other-st...,5122,2013-07-17 00:00:00,NaN,National Highway 29 Between Patkai Bridge And ...,Fresh mudslides were reported in some parts of...,National Highway 29 Between Patkai Bridge And ...,50km,mudslide,...,02/15/2018 03:51:00 PM,93.941300,25.765800,5.65,24.17,83.82,142.35,32.24,808.0,44.227692
2,The Hindu,http://www.thehindu.com/news/national/other-st...,5123,2013-07-17 00:00:00,NaN,"National Highway 2 Near Meriema Village, 10 Km...",Fresh mudslides were reported in some parts of...,"National Highway 2 Near Meriema Village, 10 Km...",50km,mudslide,...,02/15/2018 03:51:00 PM,94.088500,25.713100,3.66,25.34,116.75,188.57,51.78,1413.0,25.132660
3,E-PAO,http://e-pao.net/GP.asp?src=21..020815.aug15,7370,2015-08-01 00:00:00,NaN,between Moltuk to New Songjang village,one landslide between Moltuk to New Songjang v...,between Moltuk to New Songjang village,5km,landslide,...,02/15/2018 03:51:00 PM,94.107800,24.141900,2.09,71.58,85.55,145.97,50.65,1171.0,33.371065
4,www.assamtribune.com,http://www.assamtribune.com/scripts/detailsnew...,5084,2013-07-12 23:00:00,NaN,National Highway 2 Near Phesama Village Close ...,Inter-State bus services in Manipur have been ...,National Highway 2 Near Phesama Village Close ...,10km,landslide,...,02/15/2018 03:51:00 PM,94.116700,25.616000,18.68,107.07,129.82,199.35,51.78,1589.0,29.566317


In [26]:
cols_to_drop = [
    "photo_link", "notes", "storm_name",          
    "event_import_source", "event_import_id",      
    "submitted_date", "created_date", "last_edited_date",  
]

In [27]:
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

In [28]:
CITATION_ONLY_COLS = ["source_name", "source_link"]

In [29]:
df["fatality_count"] = df["fatality_count"].fillna(0)
df["injury_count"] = df["injury_count"].fillna(0)

In [30]:
df["location_accuracy_km"] = (
    df["location_accuracy"]
    .astype(str)
    .str.replace("km", "", case=False, regex=False)
    .str.strip()
)
df["location_accuracy_km"] = pd.to_numeric(df["location_accuracy_km"], errors="coerce")
df = df.drop(columns=["location_accuracy"])

KeyError: 'location_accuracy'

In [ ]:
df = df.sort_values("location_accuracy_km", ascending=True).reset_index(drop=True)

In [21]:
rain_triggers = ["downpour", "rain", "continuous_rain", "monsoon"]
df["is_rain_triggered"] = (
    df["landslide_trigger"].astype(str).str.lower().isin(rain_triggers)
)

In [22]:
print("Trigger value counts (check for any non-rain triggers you should exclude):")
print(df["landslide_trigger"].value_counts(dropna=False))

Trigger value counts (check for any non-rain triggers you should exclude):
landslide_trigger
downpour           167
rain                80
continuous_rain     69
monsoon             17
unknown             15
mining               3
Name: count, dtype: int64


In [23]:
size_map = {"small": 1, "medium": 2, "large": 3}
df["landslide_size_encoded"] = df["landslide_size"].astype(str).str.lower().map(size_map)

In [24]:
print("\nShape after cleaning:", df.shape)
print("\nMissing values per column:")
print(df.isna().sum().sort_values(ascending=False))
print("\nRows with is_rain_triggered == False (review these manually):")
print(df.loc[~df["is_rain_triggered"], ["event_id", "landslide_trigger", "event_title"]])


Shape after cleaning: (351, 31)

Missing values per column:
location_accuracy_km         23
landslide_size_encoded        5
source_link                   2
location_description          2
event_description             1
rainfall_7d                   0
longitude                     0
latitude                      0
rainfall_1d                   0
rainfall_3d                   0
max_rainfall_7d               0
rainfall_14d                  0
gazeteer_closest_point        0
elevation_m                   0
slope_deg                     0
is_rain_triggered             0
gazeteer_distance             0
source_name                   0
admin_division_population     0
country_code                  0
country_name                  0
injury_count                  0
fatality_count                0
landslide_setting             0
landslide_size                0
landslide_trigger             0
landslide_category            0
event_title                   0
event_date                    0
event_id   

In [31]:
df 

,source_name,source_link,event_id,event_date,event_title,event_description,location_description,landslide_category,landslide_trigger,landslide_size,...,rainfall_1d,rainfall_3d,rainfall_7d,rainfall_14d,max_rainfall_7d,elevation_m,slope_deg,location_accuracy_km,is_rain_triggered,landslide_size_encoded
0,morungexpress,http://www.morungexpress.com/local/56264.html,2587,2010-10-10 06:00:00,"Lower Chandmari area, PWD department, Kohima, ...","Today, a government quarter belonging to PWD d...","Lower Chandmari area, PWD department, Kohima, ...",landslide,downpour,medium,...,0.87,76.09,120.34,131.06,52.10,1382.0,9.350343,1.0,True,2.0
1,The Shillong Times,http://www.theshillongtimes.com/2015/06/17/ent...,6969,2015-06-16 06:15:00,"Nakham Bazaar, Tura, Meghalaya",Huge tons of mud and boulders came crashing do...,"Nakham Bazaar, Tura, Meghalaya",landslide,downpour,medium,...,27.16,62.64,230.34,245.52,61.76,306.0,8.213794,1.0,True,2.0
2,e-pao.net,http://e-pao.net/GP.asp?src=Snipp8..080911.sep11,3965,2011-09-06 08:00:00,NH 37 at Awang Khul,A massive landslide along the NH 37 at Awang K...,NH 37 at Awang Khul,landslide,rain,medium,...,22.29,30.58,45.24,61.51,22.29,39.0,5.900698,1.0,True,2.0
3,gsi.gov,http://www.portal.gsi.gov.in/portal/page?_page...,2205,2010-08-06 00:00:00,"Dawrpui Vengthar cemetary, Aizwal, Mizoram",As per PTI report a massive landslide triggere...,"Dawrpui Vengthar cemetary, Aizwal, Mizoram",landslide,downpour,medium,...,0.38,16.06,42.25,101.53,15.24,1024.0,35.939842,1.0,True,2.0
4,www.indiatvnews.com,http://www.indiatvnews.com/news/india/sikkim-l...,4990,2013-06-27 23:00:00,"Near Changu Lake, 15Km Away From Nathu La, On ...",The Mountain Black Cat Division of the Indian ...,"Near Changu Lake, 15Km Away From Nathu La, On ...",landslide,downpour,medium,...,10.50,37.88,89.33,108.19,20.53,3756.0,12.964868,1.0,True,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
346,India times,http://timesofindia.indiatimes.com/India/Flood...,259,2007-09-11 00:00:00,"Assam-Agartala National Highway, Sonarpur in J...","on highway, blocking road","Assam-Agartala National Highway, Sonarpur in J...",landslide,rain,medium,...,7.02,51.55,272.63,362.22,88.97,1346.0,10.988567,NaN,True,2.0
347,Assam Times,http://www.assamtimes.org/node/11181,6112,2014-06-28 09:00:00,"Kharghuli Hills, Assam",Over 50 people had a narrow escape in Guwahati...,"Kharghuli Hills, Assam",landslide,rain,medium,...,10.84,31.92,109.10,284.38,45.00,79.0,12.910612,NaN,True,2.0
348,thenewsofindia,NaN,3932,2011-08-18 00:00:00,Botsa and Chiechama,affected area is the NH-61 between Botsa and C...,Botsa and Chiechama,landslide,monsoon,medium,...,4.67,37.16,91.83,180.17,24.98,1014.0,8.821090,NaN,True,2.0
349,e-pao.net,http://e-pao.net/GP.asp?src=4..230711.jul11,3831,2011-07-22 00:00:00,Heibok Hills,Mudslide alarm at Heibok hills Source: The San...,Heibok Hills,landslide,downpour,medium,...,6.93,43.74,140.61,213.93,37.74,780.0,2.754582,NaN,True,2.0


In [32]:
df.to_csv("NER_landslide_cleaned_full.csv", index=False)